In [ ]:
# Check GPU and install the only missing Colab dependency
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

# pydicom is needed for DICOM (.dcm) support in IndustrialAnomalyDataset.
# All other deps (torch, torchvision, sklearn, cv2, matplotlib, numpy) are pre-installed in Colab.
!pip install -q pydicom

In [ ]:
import os

# Clone only if not already present (safe to re-run)
if not os.path.exists("anomalydino-dpmm"):
    !git clone https://github.com/Christio02/anomalydino-dpmm.git
else:
    print("Repo already cloned. Pulling latest changes...")
    !git -C anomalydino-dpmm pull

# Change into the project directory (idempotent)
if os.getcwd().split("/")[-1] != "anomalydino-dpmm":
    %cd anomalydino-dpmm

In [ ]:
# run_pipeline.py handles everything end-to-end:
#   1. Auto-downloads MVTec Grid dataset (~300 MB) to dataset/grid/ if not present
#   2. Extracts DINOv2 ViT-S/14 patch features from 264 normal training images
#   3. Fits a variational DP-GMM (BayesianGaussianMixture, max 150 components)
#   4. Scores 5 anomalous test images and saves heatmaps to figures/
!python run_pipeline.py

In [ ]:
# Display the saved anomaly heatmaps inline
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig_paths = sorted(glob.glob("figures/defect_*.png"))
if not fig_paths:
    print("No figures found. Check that run_pipeline.py completed successfully.")
else:
    fig, axes = plt.subplots(1, len(fig_paths), figsize=(6 * len(fig_paths), 5))
    if len(fig_paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, fig_paths):
        ax.imshow(mpimg.imread(p))
        ax.axis("off")
    plt.tight_layout()
    plt.show()